# VGGT → 3D Mesh

VGGT (*Visual Geometry Grounded Transformer*, Meta AI + Oxford VGG, CVPR 2025) adalah satu jaringan transformer
yang dalam **satu kali forward pass** (tanpa optimasi iteratif seperti COLMAP) langsung menebak geometri 3D
dari sekumpulan foto.


| # | Head | Key output | Shape | Isi |
|---|---|---|---|---|
| 1 | **Camera** | `pose_enc` | `(B, S, 9)` | posisi, rotasi, dan field of view tiap kamera dalam bentuk ringkas |
| 2 | **Depth** | `depth`, `depth_conf` | `(B, S, H, W, 1)`, `(B, S, H, W)` | jarak tiap piksel ke kamera + tingkat keyakinan model |
| 3 | **Point** | `world_points`, `world_points_conf` | `(B, S, H, W, 3)`, `(B, S, H, W)` | koordinat XYZ tiap piksel langsung di ruang dunia + keyakinannya |
| 4 | **Track** | `track`, `vis`, `conf` | `(B, S, N, 2)`, … | posisi titik tertentu (yang kita pilih) di setiap frame, untuk *point tracking* |

**Bedanya:**
- **Depth** bersifat **per kamera**: angka di tiap piksel hanya berarti "seberapa jauh dari kamera ini". Agar jadi 3D, depth perlu digabung dengan kamera (head 1).
- **Point** langsung memberi XYZ di **dunia bersama**. Praktis, tapi menurut paper VGGT, **depth + kamera lebih akurat** daripada point map langsung.
- **Track** hanya dipakai jika kita memberi `query_points` (titik yang ingin dilacak). Untuk mesh tidak diperlukan.

**Confidence (`*_conf`)**: nilai ≥ 1, makin besar makin yakin. Area langit, pantulan, atau tepi objek biasanya rendah.
Nilai ini kita pakai untuk membuang titik yang meragukan.

**Dunia (world frame)**: koordinat dunia = koordinat **kamera foto pertama** di list. Skalanya **relatif** (bukan meter).
Konvensinya OpenCV: **x ke kanan, y ke bawah, z ke depan**.


In [ ]:
import os, sys
!git clone -q https://github.com/facebookresearch/vggt.git
sys.path.insert(0, os.path.abspath("vggt"))
!pip install -q einops huggingface_hub safetensors open3d trimesh

In [ ]:
import glob
import numpy as np, torch, open3d as o3d, trimesh
from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
from vggt.utils.pose_enc import pose_encoding_to_extri_intri
from vggt.utils.geometry import unproject_depth_map_to_point_map

paths = glob.glob("./input/*")
os.makedirs("output", exist_ok=True)

dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
model = VGGT.from_pretrained("facebook/VGGT-1B").to("cuda").eval()
model.track_head = None   # tidak dipakai
model.point_head = None   # tidak dipakai (point cloud diambil dari depth + kamera)
torch.set_grad_enabled(False)

**Input**: `images = load_and_preprocess_images(paths)` → `(S, 3, H, W)`, nilai `[0, 1]`
- W = 518, H ikut rasio lalu dibulatkan ke kelipatan 14 (ukuran patch VGGT), maks 518 (sisanya crop tengah)
- Contoh foto 4:3 → `(S, 3, 392, 518)`

**Output model**: `pred = model(images)`. Model menambah batch `B = 1` di depan.
- `pred["pose_enc"]` → `(1, S, 9)`: parameter kamera ringkas
- `pred["pose_enc_list"]` → list `(1, S, 9)`: hasil tiap iterasi refinement; `pose_enc` = yang terakhir
- `pred["depth"]` → `(1, S, H, W, 1)`: jarak tiap piksel ke kameranya
- `pred["depth_conf"]` → `(1, S, H, W)`: keyakinan depth (≥ 1, makin besar makin yakin)
- `pred["images"]` → `(1, S, 3, H, W)`: input, disimpan untuk visualisasi
- `world_points` dan `track` tidak muncul karena head-nya dimatikan

**Isi `pose_enc` (9 angka per kamera)**
- `[0:3]` translasi t
- `[3:7]` rotasi (quaternion)
- `[7:9]` FoV vertikal, horizontal (radian)

**`extr, intr = pose_encoding_to_extri_intri(pred["pose_enc"], images.shape[-2:])`**
- `extr` → `(1, S, 3, 4)` = `[R | t]`: dunia → kamera, `X_cam = R · X_world + t`
- `intr` → `(1, S, 3, 3)` = `[[fx, 0, cx], [0, fy, cy], [0, 0, 1]]`: focal length & titik tengah, dalam piksel
- `images.shape[-2:]` = `(H, W)`. FoV berupa sudut; untuk mengubahnya ke piksel butuh ukuran gambar:
  `fx = (W/2) / tan(fov_w/2)`, `fy = (H/2) / tan(fov_h/2)`, `cx = W/2`, `cy = H/2`

**Ke numpy**: `[0]` buang batch → `.float()` (numpy tidak dukung bfloat16) → `.cpu().numpy()`
- `depth`: `(1, S, H, W, 1)` → `(S, H, W, 1)`
- `conf`: `(1, S, H, W)` → `(S, H, W)`
- `extr`: `(1, S, 3, 4)` → `(S, 3, 4)`
- `intr`: `(1, S, 3, 3)` → `(S, 3, 3)`
- `rgb`: `images` `(S, 3, H, W)` → `permute(0, 2, 3, 1)` → `(S, H, W, 3)`. Channel dipindah ke belakang supaya warna sejajar per piksel dengan `depth`.

In [ ]:
images = load_and_preprocess_images(paths, mode="pad").to("cuda")   # (S, 3, H, W)

with torch.autocast("cuda", dtype=dtype):
    pred = model(images)

extr, intr = pose_encoding_to_extri_intri(pred["pose_enc"], images.shape[-2:])   # (1,S,3,4), (1,S,3,3)
depth = pred["depth"][0].float().cpu().numpy()        # (S,H,W,1)
conf  = pred["depth_conf"][0].float().cpu().numpy()   # (S,H,W)
extr  = extr[0].float().cpu().numpy()                 # (S,3,4)
intr  = intr[0].float().cpu().numpy()                 # (S,3,3)
rgb   = images.permute(0, 2, 3, 1).cpu().numpy()      # (S,H,W,3)

### `unproject_depth_map_to_point_map(depth, extr, intr)` → `(S, H, W, 3)`
Mengubah setiap piksel menjadi titik 3D di dunia, dalam 2 langkah:
1. **Piksel → kamera** (pakai intrinsic): piksel `(u, v)` dengan kedalaman `d` menjadi
   $X_{cam} = \left(\frac{(u-c_x)\,d}{f_x},\ \frac{(v-c_y)\,d}{f_y},\ d\right)$
2. **Kamera → dunia** (pakai extrinsic, dibalik): $X_{world} = R^\top (X_{cam} - t)$

Hasilnya satu XYZ untuk **setiap piksel di setiap foto**.

### Filter confidence
`conf > np.percentile(conf, 50)` → mask boolean `(S, H, W)`, `True` untuk 50% piksel paling yakin.
`pts[keep]` mengambil hanya titik yang `True` sehingga hasilnya **pipih** menjadi `(N, 3)` (daftar titik).
Angka `50` bisa diubah: lebih besar = lebih bersih tapi lebih bolong.

### Posisi kamera: `cams = -einsum("sji,sj->si", R, t)`
Pusat kamera di dunia adalah $C = -R^\top t$. `einsum` menghitungnya untuk semua S kamera sekaligus
(`"sji,sj->si"` = transpose R lalu kalikan dengan t, per kamera). Hasil `(S, 3)`.

### Arah ke kamera disimpan di slot normal (trik)
`cams[:, None, None] - pts` → `(S, H, W, 3)`: vektor dari tiap titik ke kamera yang memotretnya
(`[:, None, None]` menambah dua dimensi supaya `(S,3)` bisa dikurangkan dengan `(S,H,W,3)`, *broadcasting*).
Vektor ini disimpan sementara di `pcd.normals`. Gunanya baru terasa di langkah 5: menentukan sisi luar permukaan.

### Bersih-bersih
- `voxel_down_sample(size/1000)`: ruang dibagi kubus kecil (1/1000 ukuran scene); titik dalam satu kubus dirata-rata menjadi satu titik.
  Jumlah titik turun drastis (jutaan → ratusan ribu) tanpa kehilangan bentuk. Warna dan "normal" ikut dirata-rata.
- `remove_statistical_outlier(20, 2.0)`: buang titik yang jarak rata-rata ke 20 tetangganya > 2 standar deviasi (titik "melayang").

In [ ]:
pts = unproject_depth_map_to_point_map(depth, extr, intr)   # (S,H,W,3)
keep = conf > np.percentile(conf, 50)                       # buang 50% titik paling tidak yakin

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(pts[keep].astype(np.float64))
pcd.colors = o3d.utility.Vector3dVector(rgb[keep].astype(np.float64))

# simpan arah "titik -> kamera" sementara di slot normal (dipakai di langkah mesh)
cams = -np.einsum("sji,sj->si", extr[:, :, :3], extr[:, :, 3])   # posisi tiap kamera
pcd.normals = o3d.utility.Vector3dVector((cams[:, None, None] - pts)[keep].astype(np.float64))

size = np.linalg.norm(pcd.get_max_bound() - pcd.get_min_bound())
pcd = pcd.voxel_down_sample(size / 1000)
pcd, _ = pcd.remove_statistical_outlier(20, 2.0)

### Normal
Poisson butuh **normal** (arah tegak lurus permukaan) di setiap titik, dan arahnya harus **ke luar**.
- `estimate_normals()` menghitung normal dari tetangga terdekat. Arahnya benar, tapi **bisa terbalik** (ke dalam atau ke luar acak).
- Perbaikannya: bandingkan dengan vektor "titik → kamera" yang tadi disimpan (`view`).
  Permukaan yang terlihat kamera pasti menghadap kamera, jadi jika `n · view < 0` (berlawanan arah), normal dibalik (`*= -1`).

### Poisson
- `depth=9`: resolusi grid internal (2⁹ = 512 per sisi). 8 = kasar/cepat, 10–11 = detail/berat.
- `dens`: seberapa banyak titik asli yang mendukung tiap vertex. Poisson selalu menghasilkan permukaan tertutup,
  jadi di area tanpa data ia "mengarang" kulit tipis. Vertex dengan densitas 5% terendah dibuang.
- `crop(bounding box)`: potong bagian mesh yang menggelembung keluar dari area point cloud.
- Warna vertex otomatis diinterpolasi dari warna point cloud.

In [ ]:
# hitung normal, lalu hadapkan ke kamera yang melihat titik itu
view = np.asarray(pcd.normals).copy()
pcd.estimate_normals()
n = np.asarray(pcd.normals)
n[(n * view).sum(1) < 0] *= -1
pcd.normals = o3d.utility.Vector3dVector(n)

mesh, dens = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=9)
dens = np.asarray(dens)
mesh.remove_vertices_by_mask(dens < np.quantile(dens, 0.05))   # buang permukaan "ngarang"
mesh = mesh.crop(pcd.get_axis_aligned_bounding_box())

## 6. Simpan ke GLB

- **Konversi sumbu:** VGGT memakai OpenCV (y **ke bawah**, z **ke depan**), sedangkan glTF/three.js/Blender memakai
  y **ke atas** dan kamera melihat ke −z. Mengalikan `[1, -1, -1]` membalik sumbu y dan z sehingga model tidak terbalik saat dibuka.
- **Warna:** Open3D menyimpan warna sebagai float `[0, 1]`, sedangkan GLB memakai `uint8` `[0, 255]`, jadi dikali 255.
- Hasil: `output/mesh.glb`, siap dirender interaktif (three.js, `<model-viewer>`, Blender).

In [ ]:
out = trimesh.Trimesh(
    vertices=np.asarray(mesh.vertices) * [1, -1, -1],   # OpenCV -> y ke atas
    faces=np.asarray(mesh.triangles),
    vertex_colors=(np.asarray(mesh.vertex_colors) * 255).astype(np.uint8),
)
out.export("output/mesh.glb")